# medwaves driver

Runs each stage of the pipeline on Colab. Every stage can be re-run and skips work already done; all data, caches and checkpoints live under `DATA_ROOT` on Google Drive.

Before running, add these in **Colab Secrets** (key icon in the left sidebar) and allow notebook access:

- `COPERNICUSMARINE_SERVICE_USERNAME`, `COPERNICUSMARINE_SERVICE_PASSWORD`: Copernicus Marine account
- `CDSAPI_KEY`: Climate Data Store personal access token (for ERA5)

Use a T4 GPU runtime for training (Runtime → Change runtime type).

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/medwaves"
os.makedirs(DATA_ROOT, exist_ok=True)
os.environ["MEDWAVES_DATA_ROOT"] = DATA_ROOT

In [ ]:
REPO_URL = "https://github.com/ellen-lyons/medwaves.git"
REPO_DIR = "/content/medwaves"
BRANCH = "main"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch -q origin
    !git -C {REPO_DIR} checkout -q {BRANCH}
    !git -C {REPO_DIR} pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# Editable install; keeps Colab's preinstalled CUDA torch.
!pip install -q -e {REPO_DIR}

import torch, torch_geometric
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| PyG", torch_geometric.__version__)

In [ ]:
# Credentials go into environment variables only: never printed, never written to the repo.
from google.colab import userdata

for name in ["COPERNICUSMARINE_SERVICE_USERNAME", "COPERNICUSMARINE_SERVICE_PASSWORD", "CDSAPI_KEY"]:
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: set")
    except Exception as e:
        print(f"{name}: not available ({type(e).__name__})")
os.environ.setdefault("CDSAPI_URL", "https://cds.climate.copernicus.eu/api")

In [ ]:
# Copy processed data (a few large files) from Drive to fast local disk; stages read from here
# and still write to Drive. Re-run after a stage writes new processed data.
import shutil
from pathlib import Path

LOCAL_ROOT = "/content/medwaves_local"
src, dst = Path(DATA_ROOT) / "processed", Path(LOCAL_ROOT) / "processed"
copied = 0
if src.exists():
    for f in sorted(p for p in src.rglob("*") if p.is_file()):
        out = dst / f.relative_to(src)
        if not out.exists() or out.stat().st_size != f.stat().st_size or out.stat().st_mtime < f.stat().st_mtime:
            out.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, out)
            copied += 1
print(f"copied {copied} file(s) from {src} to {dst}")
os.environ["MEDWAVES_LOCAL_ROOT"] = LOCAL_ROOT

In [ ]:
!medwaves-check-splits

## Stages

Run one at a time; check each stage's report before moving on.

In [ ]:
# Stage 1, first: download one day of each source and print its format (columns, QC codes,
# platform types, mission codes). Check it looks as expected before the full run.
os.environ["MEDWAVES_SCRATCH"] = "/content/medwaves_scratch"   # whole global altimeter files, local disk
!medwaves-data --probe

In [ ]:
# Stage 1: data pipeline (pilot: 2019, Sentinel-3A + held-out Sentinel-3B). Re-run after a
# disconnect: finished downloads and steps are skipped. Prints the data report at the end
# (also saved to DATA_ROOT/reports/stage1_2019.md).
!medwaves-data

### After stage 1: commit the buoy split

Stage 1 draws the held-out buoys once (whole 30 km clusters, stratified by region, only buoys with 2023 data) and writes them to `DATA_ROOT/splits/heldout_buoys.yaml` on Drive. It is not used until it is in the repo, and it is never redrawn (to redraw, delete it by hand). Until then the stores hold altimetry only.

1. Download `MyDrive/medwaves/splits/heldout_buoys.yaml` from Drive (or print it with the cell below).
2. Review it, copy it to `configs/heldout_buoys.yaml` in your local clone, then
   `git add configs/heldout_buoys.yaml && git commit -m "Commit held-out buoy split" && git push`.
3. Re-run the clone/pull cell above, then the check below: it should report the train-pool and held-out buoy counts.
4. Re-run the stage 1 cell: it only rebuilds the stores (now with buoys) and the report.

In [ ]:
!cat {DATA_ROOT}/splits/heldout_buoys.yaml

In [ ]:
!medwaves-check-splits

In [ ]:
# Stage 2: baselines and reanalysis comparison
!medwaves-baselines

In [ ]:
# Stage 3: training (checkpoints every epoch, resumes automatically)
!medwaves-train

In [ ]:
# Stage 4: evaluation
!medwaves-eval